- 근데... 주식 데이터는 차분말고 없지 않아요? SARIMA는 찾아봐야겠고.

# 데이터 입수

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ibrahimshahrukh/google-alphabet-stock-prices-2016-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA # 아리마
from statsmodels.tsa.statespace.sarimax import SARIMAX # ...이거 뭐라고 읽음?
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf # 우리 이거 볼거예요 예
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore') # 모든 경고 무시

In [ ]:
alphabet_df = pd.read_csv(f'{path}/microsoft_stock_prices_2016_2026_refined.csv')
# 이건 구글이냐 마이크로소프트냐 한 35초 고민함

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="tab10", style="whitegrid", font_scale=1)
sns.color_palette("tab10", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Galmuri11'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

# 데이터 확인

In [ ]:
alphabet_df.head()
# 그... 어떤게 종가임?

In [ ]:
alphabet_df.isna().sum()

In [ ]:
alphabet_df.shape

# 전처리
- 종가만 따로 빼서 시리즈로 만들고 차분할거예요.
- 시계열 데이터는 차분을 해야 합니다.

In [ ]:
# 날짜를 인덱스로
alphabet_df['Date'] = pd.to_datetime(alphabet_df['Date']) # 날짜 형식 확인
alphabet_df.set_index('Date', inplace=True)

In [ ]:
alphabet_series = alphabet_df['Close'] # 니가 종가구나
alphabet_series

alphabet_diff = alphabet_series.diff().dropna()

# ARIMA

## ACF, PACF

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (15,5)) # 사실 이게 뭔 기준인지 아직도 모르겠음둥

plot_acf(alphabet_diff, ax = ax[0]) # 위에서 둘로 쨌으니까 한쪽에는 ACF가 들어가고
ax[0].set_title('ACF')

plot_pacf(alphabet_diff, ax = ax[1]) # 얘는 PACF임다
ax[1].set_title('PACF')

plt.show()

- 모든 주가 데이터가 다 그렇듯이 acf, pacf 엘보가 0임...

## ARIMA (p, d, q)

In [ ]:
model = ARIMA(alphabet_series, order = (0,1,0)) # p, d, q = 0, 1, 0
model_fit = model.fit()

model_fit.plot_diagnostics(figsize=(12, 8)) # 잔차 분석(뭔지 모르겠음...)
plt.show()
# 경고 왜뜸?

- 경고문 뭐여...

## Forecast

In [ ]:
forecast_steps = 10
# 상세 예측 객체 가져오기
forecast_obj = model_fit.get_forecast(steps=forecast_steps)

# 예측값 (평균)
mean_forecast = forecast_obj.predicted_mean

# 신뢰구간 (95% 자신감으로 이 범위 안에 있음)
confidence_intervals = forecast_obj.conf_int()

print(mean_forecast) # 예측 평균
print(confidence_intervals) # 음 이정도 될듯?

In [ ]:
# 시각화 사이즈 설정
plt.figure(figsize=(12, 6))

# 1. 실제 데이터 (최근 50일치만 보기)
plt.plot(alphabet_df.index[-50:], alphabet_df['Close'][-50:], label='Actual')

# 2. 예측 데이터 날짜 생성 (마지막 날짜 다음 영업일부터 시작)
last_date = alphabet_df.index[-1]
forecast_index = pd.date_range(start=last_date, periods=forecast_steps + 1, freq='B')[1:]

# 3. 예측값 그리기
plt.plot(forecast_index, mean_forecast, color='red', label='Forecast')

# 4. 신뢰구간 색칠하기
plt.fill_between(forecast_index,
                 confidence_intervals.iloc[:, 0],
                 confidence_intervals.iloc[:, 1],
                 color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title('주가 예측 결과')
plt.legend()
plt.show()

# SARIMA
- ARIMA에 계절을 추가해보세요!

In [ ]:
model = SARIMAX(alphabet_series, seasonal_order=(1, 1, 1, 5)) # p, d, q = 0, 1, 0
model_fit = model.fit()

model_fit.plot_diagnostics(figsize=(12, 8)) # 잔차 분석(뭔지 모르겠음...)
plt.show()
# 이건 0, 1, 0보다 1, 1, 1이 더 잘나온다.

## Forecast

In [ ]:
# 상세 예측 객체 가져오기
forecast_obj = model_fit.get_forecast(steps=forecast_steps)

# 예측값 (평균)
mean_forecast = forecast_obj.predicted_mean

# 신뢰구간 (95% 자신감으로 이 범위 안에 있음)
confidence_intervals = forecast_obj.conf_int()

print(mean_forecast) # 예측 평균
print(confidence_intervals) # 음 이정도 될듯?

In [ ]:
# 시각화 사이즈 설정
plt.figure(figsize=(12, 6))

# 1. 실제 데이터 (최근 50일치만 보기)
plt.plot(alphabet_df.index[-50:], alphabet_df['Close'][-50:], label='Actual')

# 2. 예측 데이터 날짜 생성 (마지막 날짜 다음 영업일부터 시작)
last_date = alphabet_df.index[-1]
forecast_index = pd.date_range(start=last_date, periods=forecast_steps + 1, freq='B')[1:]

# 3. 예측값 그리기
plt.plot(forecast_index, mean_forecast, color='red', label='Forecast')

# 4. 신뢰구간 색칠하기
plt.fill_between(forecast_index,
                 confidence_intervals.iloc[:, 0],
                 confidence_intervals.iloc[:, 1],
                 color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title('주가 예측 결과')
plt.legend()
plt.show()

# ARIMAX
- ARIMA에 외부 변수를 싸서 드셔보세요! ~~뭔 소리야~~
- 데이터에 없어서 걍 만들어서 하긴 했는데, 주식으로 ARIMAX 하실거면 나스닥이든 코스닥이든 그 주가에 맞는 지수가 있으면 좋습니다. 

In [ ]:
# 예시: 간단하게 외부 변수(exog) 생성하기
alphabet_df['MA5'] = alphabet_df['Close'].rolling(window=5).mean() # 5일 이동평균
alphabet_df['Vol_Change'] = alphabet_df['Volume'].pct_change()   # 거래량 변화율
alphabet_df['Day'] = alphabet_df.index.dayofweek                # 요일 (0=월, 6=일)

# 결측치 제거 후 모델에 투입
exog = alphabet_df[['MA5', 'Vol_Change', 'Day']].dropna()
target = alphabet_df['Close'].loc[exog.index]

In [ ]:
model = SARIMAX(target, exog=exog, order=(0,1,0), seasonal_order=(1,1,1,5))
model_fit = model.fit()

model_fit.plot_diagnostics(figsize=(12, 8)) # 잔차 분석(뭔지 모르겠음...)
plt.show()

## Forecast

In [ ]:
# (대충 미래 예측 데이터라는 얘기)
last_exog = exog.iloc[-1:] # 마지막 날의 외부 변수들
future_exog = pd.concat([last_exog] * forecast_steps, ignore_index=True)

# 상세 예측 객체 가져오기
forecast_obj = model_fit.get_forecast(steps=forecast_steps, exog=future_exog)

# 예측값 (평균)
mean_forecast = forecast_obj.predicted_mean

# 신뢰구간 (95% 자신감으로 이 범위 안에 있음)
confidence_intervals = forecast_obj.conf_int()

print(mean_forecast) # 예측 평균
print(confidence_intervals) # 음 이정도 될듯?

In [ ]:
# 시각화 사이즈 설정
plt.figure(figsize=(12, 6))

# 1. 실제 데이터 (최근 50일치만 보기)
plt.plot(alphabet_df.index[-50:], alphabet_df['Close'][-50:], label='Actual')

# 2. 예측 데이터 날짜 생성 (마지막 날짜 다음 영업일부터 시작)
last_date = alphabet_df.index[-1]
forecast_index = pd.date_range(start=last_date, periods=forecast_steps + 1, freq='B')[1:]

# 3. 예측값 그리기
plt.plot(forecast_index, mean_forecast, color='red', label='Forecast')

# 4. 신뢰구간 색칠하기
plt.fill_between(forecast_index,
                 confidence_intervals.iloc[:, 0],
                 confidence_intervals.iloc[:, 1],
                 color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title('주가 예측 결과')
plt.legend()
plt.show()

# 모델 평가

In [ ]:
def evaluate_model(actual, forecast, model_name):
    # 1. 인덱스 무시하고 값만 추출 (numpy array로 변환)
    # 실제값과 예측값의 '순서'만 같으면 비교가 가능해집니다.
    a = np.array(actual).flatten()
    f = np.array(forecast).flatten()

    # 2. 혹시나 모를 개수 차이 조정 (짧은 쪽에 맞춤)
    min_len = min(len(a), len(f))
    a = a[:min_len]
    f = f[:min_len]

    if min_len == 0:
        print(f"[{model_name}] 에러: 비교할 데이터가 아예 없습니다.")
        return

    # 3. 지표 계산
    rmse = np.sqrt(mean_squared_error(a, f))
    mae = mean_absolute_error(a, f)
    mape = np.mean(np.abs((a - f) / a)) * 100

    print(f"[{model_name} 성적표]")
    print(f"  - RMSE: {rmse:.2f}")
    print(f"  - MAE : {mae:.2f}")
    print(f"  - MAPE: {mape:.2f}%")
    print("-" * 35)

In [ ]:
# 1. 데이터 준비 (최근 20일 데이터를 '정답지'로 떼어놓기)
test_size = 20
train = alphabet_series[:-test_size]
test = alphabet_series[-test_size:]

# (ARIMAX용 외부변수도 동일하게 분할)
train_exog = exog[:-test_size]
test_exog = exog[-test_size:]

# 1. 두 데이터의 공통 날짜 인덱스 추출
common_index = train.index.intersection(train_exog.index)

# 2. 공통 인덱스로만 다시 필터링 (행 개수와 날짜를 완전히 일치시킴)
train_final = train.loc[common_index]
train_exog_final = train_exog.loc[common_index]
# ---------------------------------------------------------
# 2. 각 모델 학습 및 예측
# ---------------------------------------------------------

# (1) ARIMA (그대로)
model_arima = ARIMA(train, order=(0, 1, 0)).fit()
pred_arima = model_arima.forecast(steps=test_size)

# (2) SARIMA (그대로)
model_sarima = SARIMAX(train, order=(0, 1, 0), seasonal_order=(1, 1, 1, 5)).fit()
pred_sarima = model_sarima.forecast(steps=test_size)

# (3) ARIMAX (★이 부분을 교체하세요★)
# 인덱스가 맞춰진 train_final, train_exog_final 사용
model_arimax = SARIMAX(train_final,
                       exog=train_exog_final,
                       order=(0, 1, 0),
                       seasonal_order=(1, 1, 1, 5)).fit()

# 테스트 데이터도 인덱스를 맞춘 후 예측
common_test_index = test.index.intersection(test_exog.index)
test_final = test.loc[common_test_index]
test_exog_final = test_exog.loc[common_test_index]

pred_arimax = model_arimax.get_forecast(steps=len(test_final),
                                        exog=test_exog_final).predicted_mean

# ---------------------------------------------------------
# 3. 드디어 평가!
# ---------------------------------------------------------
evaluate_model(test, pred_arima, "ARIMA")
evaluate_model(test, pred_sarima, "SARIMA")
evaluate_model(test_final, pred_arimax, "ARIMAX") # 정렬된 test_final 사용

In [ ]:
def get_model_stats(model_fit, name):
    print(f"[{name} 내부 평가 지표]")
    print(f"  - AIC: {model_fit.aic:.2f}")
    print(f"  - BIC: {model_fit.bic:.2f}")
    print(f"  - Log Likelihood: {model_fit.llf:.2f}")
    print("-" * 35)

get_model_stats(model_arima, "ARIMA")
get_model_stats(model_sarima, "SARIMA")
get_model_stats(model_arimax, "ARIMAX")

## Summary

In [ ]:
model_arima.summary()

In [ ]:
model_sarima.summary()

In [ ]:
model_arimax.summary()